# Lights On: Keep the Hostels Powered
HackSphere | Aayam XI | Problem 2

**Task:** predict the hourly electricity use (`load_kwh`) of 34 hostel blocks from 5 Oct to 4 Dec 2025.

**Important thing about the score (BWE):** if we predict too LOW we get punished 3 times more than if we predict too HIGH.
So we don't want the average prediction, we want a slightly higher (safer) one.
For this we train the model to predict the **75th percentile** instead of the mean.

**Steps in this notebook**
1. Load the data
2. Remove wrong sensor readings
3. Look at the data (graph 1)
4. Make features
5. Check the model on the last 61 days of train data (graph 2)
6. Train on everything and save `submission.csv`

## Step 0: Imports and settings
Here we import the libraries we need. `train.csv`, `test.csv` and `sample_submission.csv` should be in the same
folder as this notebook (or change `DATA_DIR`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor

DATA_DIR = "."    # folder where the csv files are kept
ALPHA = 0.75      # 0.75 = 75th percentile. Under-forecast costs 3x, so 3 / (3 + 1) = 0.75

## Step 1: Load the data
We read the 3 csv files and convert the `timestamp` column into real dates,
so that pandas understands hour, weekday, month etc.

In [ ]:
train = pd.read_csv(DATA_DIR + "/train.csv")
test = pd.read_csv(DATA_DIR + "/test.csv")
sample = pd.read_csv(DATA_DIR + "/sample_submission.csv")

train["timestamp"] = pd.to_datetime(train["timestamp"])
test["timestamp"] = pd.to_datetime(test["timestamp"])

print("train shape:", train.shape)
print("test shape:", test.shape)
print("train dates:", train["timestamp"].min(), "to", train["timestamp"].max())
print("test dates:", test["timestamp"].min(), "to", test["timestamp"].max())
train.head()

## Step 2: Remove wrong sensor readings
If a meter says `0` or a negative number for a whole hostel, the meter is most probably broken
(a hostel can't use zero electricity). We make these values empty (NaN) so the model does not learn from them.

In [ ]:
bad = train["load_kwh"].isna() | (train["load_kwh"] <= 0)
print("wrong readings:", bad.sum(), "rows =", round(bad.mean() * 100, 2), "%")
train.loc[bad, "load_kwh"] = np.nan

## Step 3: Look at the data (Graph 1)
Before making a model we should check if there is any pattern in the data.

Different blocks have very different sizes (some use 20 kWh, some 200 kWh), so for the graph we divide each
block's load by its own average. Now **1.0 means a normal hour** for that block, 1.5 means 50% more than normal.

The graph shows the average load for every hour of the day, separately for normal weekdays, weekends and holidays.

In [ ]:
avg_load = train.groupby("building_id")["load_kwh"].transform("mean")
train["load_ratio"] = train["load_kwh"] / avg_load      # only used for this graph

hours = train["timestamp"].dt.hour
weekend = train["timestamp"].dt.dayofweek >= 5

plt.figure(figsize=(10, 5))
plt.plot(train[(train["is_holiday"] == 0) & (~weekend)].groupby(hours)["load_ratio"].mean(), label="Normal weekday")
plt.plot(train[(train["is_holiday"] == 0) & (weekend)].groupby(hours)["load_ratio"].mean(), label="Weekend")
plt.plot(train[train["is_holiday"] == 1].groupby(hours)["load_ratio"].mean(), label="Holiday")
plt.xlabel("Hour of the day")
plt.ylabel("Load / block average (1.0 = normal)")
plt.title("Average hostel load during the day")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**What we learn from graph 1:** the load changes a lot with the hour of the day and it is different on weekends and holidays.
So `hour`, `day of week` and `is_holiday` must be in our features.

## Step 4: Make features
A model can only learn from the columns we give it. These are the columns we use:

- **hour, day_of_week, month, day_of_year, is_weekend**: from the timestamp (hostels use power differently at 3 AM and at 10 PM)
- **is_holiday**: hostels become empty in semester breaks
- **weather columns**: temperature, humidity, wind etc. (the problem says test weather is a perfect forecast, so we can use it)
- **floor_area_sqm and building_age**: size and age of the building
- **building and campus** as numbers, so the model can learn how each block behaves
- **load_last_year**: the load of the same block exactly **364 days before**. 364 days = 52 weeks, so it is the same weekday.
  This is the most useful feature because last year's load already shows exam weeks and semester breaks.

**Why not use yesterday's load?** We have to predict 61 days into the future at once, so yesterday's load is not known
for most test rows. Last year's load is always known.

**Scaling:** the score treats every block equally, so we divide each block's load by that block's average (`scale`).
The model learns "how many times the normal load" and in the end we multiply back to get kWh.

In [ ]:
# last year's load: move every train timestamp 364 days forward,
# then it can be joined with the row that has that new date
last_year = train[["building_id", "timestamp", "load_kwh"]].copy()
last_year["timestamp"] = last_year["timestamp"] + pd.Timedelta(days=364)
last_year = last_year.rename(columns={"load_kwh": "load_last_year"})

# average load of each block in train (used for scaling)
scale = train.groupby("building_id")["load_kwh"].mean()

# campus name -> number
campus_number = {"CAMPUS_A": 0, "CAMPUS_B": 1, "CAMPUS_C": 2, "CAMPUS_D": 3, "CAMPUS_E": 4, "CAMPUS_F": 5}

weather_cols = ["air_temp_c", "dew_temp_c", "cloud_cover_okta", "rain_1h_mm",
                "pressure_hpa", "wind_dir_deg", "wind_speed_ms"]


def make_features(df):
    # add last year's load
    df = df.merge(last_year, on=["building_id", "timestamp"], how="left")

    # time features
    df["hour"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek    # 0 = Monday, 6 = Sunday
    df["month"] = df["timestamp"].dt.month
    df["day_of_year"] = df["timestamp"].dt.dayofyear
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

    # building features
    df["building"] = df["building_id"].str[3:].astype(int)           # HB_07 -> 7
    df["campus"] = df["campus_id"].map(campus_number)
    df["building_age"] = df["timestamp"].dt.year - df["year_built"]

    # scaling
    df["scale"] = df["building_id"].map(scale)
    df["load_last_year"] = df["load_last_year"] / df["scale"]
    return df


features = ["hour", "day_of_week", "month", "day_of_year", "is_weekend", "is_holiday",
            "floor_area_sqm", "building_age", "building", "campus", "load_last_year"] + weather_cols
cat_features = ["building", "campus"]    # these are labels, not real numbers

train = make_features(train)
test = make_features(test)
train["target"] = train["load_kwh"] / train["scale"]    # this is what the model will predict

print("number of features:", len(features))
train[features].head()

## Step 5: Score function and model check (Graph 2)
**Score function:** `bwe()` is the formula from the problem statement. For every block we add the costs
(under-forecast x3, over-forecast x1), divide by the real total load of the block, then take the average of all blocks.
Lower is better.

**Model check:** we hide the **last 61 days of train data** (same length as the real test), train on the days before it,
and score the model on the hidden days. We don't split randomly because that would let the model see the future.

We also score a simple baseline (every hour = 75th percentile load of that block) to see if our model is really better.

In [ ]:
def bwe(actual, predicted, building):
    under = actual > predicted
    cost = np.where(under, 3 * (actual - predicted), predicted - actual)
    d = pd.DataFrame({"b": building, "cost": cost, "actual": actual})
    total = d.groupby("b").sum()
    return 100 * (total["cost"] / total["actual"]).mean()


def new_model(alpha=ALPHA):
    # gradient boosting = many small decision trees, every new tree fixes the mistakes of the old ones
    return HistGradientBoostingRegressor(
        loss="quantile", quantile=alpha,      # predict the 75th percentile
        max_iter=400, learning_rate=0.05,     # number of trees and learning speed
        max_depth=8, min_samples_leaf=50,     # so the trees don't just memorise the data
        categorical_features=cat_features,
        random_state=42)


# split: last 61 days = validation
val_start = train["timestamp"].max().normalize() - pd.Timedelta(days=60)
has_target = train["target"].notna()
fit_data = train[(train["timestamp"] < val_start) & has_target]
val_data = train[(train["timestamp"] >= val_start) & has_target]

model = new_model()
model.fit(fit_data[features], fit_data["target"])

# predict, remove negatives, and convert back to kWh
val_pred = np.clip(model.predict(val_data[features]), 0, None) * val_data["scale"]

# baseline: 75th percentile load of every block
q75 = fit_data.groupby("building_id")["load_kwh"].quantile(ALPHA)
base_pred = val_data["building_id"].map(q75)

actual = val_data["load_kwh"].values
b_ids = val_data["building_id"].values
print("Validation days:", val_start.date(), "to", train["timestamp"].max().date())
print("Baseline BWE:", round(bwe(actual, base_pred.values, b_ids), 2))
print("Our model BWE:", round(bwe(actual, val_pred.values, b_ids), 2), "(lower is better)")

# ---- Graph 2: real vs predicted, average over all blocks, one point per day ----
plot_df = pd.DataFrame({
    "day": val_data["timestamp"].dt.normalize().values,
    "real": (val_data["load_kwh"] / val_data["scale"]).values,
    "pred": (val_pred / val_data["scale"]).values})
daily = plot_df.groupby("day").mean()

plt.figure(figsize=(11, 5))
plt.plot(daily.index, daily["real"], label="Real load")
plt.plot(daily.index, daily["pred"], label="Our prediction")
plt.xlabel("Date")
plt.ylabel("Load / block average (1.0 = normal)")
plt.title("Real vs predicted load on the validation days (prediction is a bit higher on purpose)")
plt.legend()
plt.grid(alpha=0.3)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**What we learn from graph 2:** our line follows the real load, and it stays a bit above it.
This is on purpose, because predicting too low is punished 3 times more.

**Why not make the line sit exactly on the real load?** A line that sits on the real load is the *average* prediction.
An average prediction is too low in about half of the hours, and every too-low hour costs 3x. So it gets a worse score.
The next block proves it: we train the same model with `alpha = 0.5` (average) and with `alpha = 0.75` (ours)
and compare the BWE on the same validation days.

In [ ]:
# same model, only the alpha is different
for a in [0.5, 0.6, 0.75]:
    m = new_model(alpha=a)
    m.fit(fit_data[features], fit_data["target"])
    p = np.clip(m.predict(val_data[features]), 0, None) * val_data["scale"]
    print("alpha =", a, "-> BWE =", round(bwe(actual, p.values, b_ids), 2))

## Step 6: Train on all data and make the submission
The approach works, so now we train a new model on **all** the train data (more data = better model),
predict the test hours, convert back to kWh and save `submission.csv` in the same order as `sample_submission.csv`.

In [ ]:
all_data = train[train["target"].notna()]
final_model = new_model()
final_model.fit(all_data[features], all_data["target"])

test["load_kwh"] = np.clip(final_model.predict(test[features]), 0, None) * test["scale"]

submission = sample[["row_id"]].merge(test[["row_id", "load_kwh"]], on="row_id", how="left")
submission.to_csv("submission.csv", index=False)

print("submission shape:", submission.shape)
print("missing values:", submission["load_kwh"].isna().sum())
submission.head()

## Done
`submission.csv` has the columns `row_id, load_kwh` with one row for each test row.

**Things we can still try:**
- Change `ALPHA` to 0.7 or 0.8 and check which gives the better validation BWE
- Add the average of last year's load over a few hours/days
- Add a feature like "days until the next holiday"